# AzLegalRAG - Colab Demo

Azerbaijani Legal Document Q&A using RAG.

**Requirements**: L4 GPU (24GB VRAM)

Go to: Runtime > Change runtime type > Select L4 GPU

## 1. Install Dependencies

In [ ]:
!pip install -q torch transformers accelerate sentence-transformers langchain langchain-community chromadb datasets tqdm

## 2. Clone Repository

In [ ]:
!git clone https://github.com/StartZer0/AzLegalRAG.git
%cd AzLegalRAG

## 3. Check GPU

In [ ]:
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 4. Ingest Documents (Run Once - Takes ~30 min)

In [ ]:
import sys
sys.path.insert(0, './src')

from ingest import load_eqanun, chunk_documents
from embed import create_vectorstore

# Load dataset
dataset = load_eqanun()
print(f"Loaded {len(dataset)} documents")

# Chunk documents
chunks = chunk_documents(dataset)
print(f"Created {len(chunks)} chunks")

In [ ]:
# Create vectorstore (this takes time!)
vectorstore = create_vectorstore(chunks)
print("Vectorstore created!")

## 5. Test Search (Without LLM)

In [ ]:
from embed import load_vectorstore
from retrieve import semantic_search

vs = load_vectorstore()
results = semantic_search(vs, "Emek muqavilesi nedir?", k=3)

for i, doc in enumerate(results, 1):
    print(f"\n[{i}] Source: {doc.metadata['source']}")
    print(doc.page_content[:300] + "...")

## 6. Load LLM & Create RAG Chain

In [ ]:
from generate import get_llm, create_rag_chain

# Load LLM (takes a few minutes)
llm = get_llm()
print("LLM loaded!")

# Create RAG chain
chain = create_rag_chain(vs, llm)
print("RAG chain ready!")

## 7. Test RAG Q&A

In [ ]:
question = "Emek muqavilesi nedir?"
result = chain({"query": question})

print("=" * 50)
print(f"Question: {question}")
print("=" * 50)
print(f"\nAnswer:\n{result['result']}")
print("\n" + "=" * 50)
print("Sources:")
for doc in result["source_documents"]:
    print(f"- {doc.metadata['source']}")

In [ ]:
# Try more questions
questions = [
    "Mehkeme qerari nece shekillendirilir?",
    "Nikah muqavilesi ucun ne teleb olunur?",
    "Vergiler hansi novlere bolunur?"
]

for q in questions:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    result = chain({"query": q})
    print(f"A: {result['result'][:500]}...")

## 8. Save to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/AzLegalRAG
!cp -r ./vectorstore /content/drive/MyDrive/AzLegalRAG/
print("Vectorstore saved to Google Drive!")